In [1]:
# ============================================================
# NEPALI REASONING DATASET GENERATION
# Shared entity names, disjoint reasoning examples
#
# Total = 12,000
# Train = 9,600
# Validation = 1,200
# Test = 1,200
# ============================================================

import json
import random
from pathlib import Path
from collections import Counter


# ============================================================
# CONFIG
# ============================================================

SEED = 42
random.seed(SEED)

TOTAL_EXAMPLES = 12000

TEMPLATES = [
    "T1",
    "T2",
    "T3",
    "T4",
    "T5",
    "T6"
]

# Per template
PER_TEMPLATE = {
    "train": 1600,
    "validation": 200,
    "test": 200,
}


# ============================================================
# SHARED NAME POOL
#
# IMPORTANT:
# Names are intentionally shared across train/val/test.
#
# What remains disjoint:
# - exact prompts
# - exact generated reasoning problems
# ============================================================

NAMES = [
    "राम",
    "सुरेश",
    "महेश",
    "अनिता",
    "सीता",
    "लता",
    "अर्जुन",
    "भरत",
    "काव्या",
    "दिव्या",
    "रमेश",
    "गोपाल",
    "स्वाति",
    "नन्दिनी",
    "हरिश",
    "मधु",
    "दीपक",
    "श्रावणी",
    "किरण",
    "विनय",
    "पद्मा",
    "दीप्ति",
    "तरुण",
    "नवीन",
    "सौम्या",
    "मेघना",
    "आदित्य",
    "वर्षा",
    "सुनिल",
    "पवन",
    "कार्तिक",
    "संदीप",
    "प्रणय",
    "रोहित",
    "मनोज",
    "विक्रम",
    "लक्ष्मी",
    "हरिका",
]


# ============================================================
# RELATION TYPES
#
# Internally:
#
# bigger > smaller
#
# ============================================================

RELATIONS = {

    "height": {

        "greater": [
            "{a} {b} भन्दा अग्लो छन्।",
            "{b} सँग तुलना गर्दा {a} अग्लो छन्।",
        ],

        "reverse": [
            "{b} {a} भन्दा होचो छन्।",
            "{a} सँग तुलना गर्दा {b} होचो छन्।",
        ],

        "greater_question":
            "{a}, {b} — यीमध्ये को अग्लो छ?",

        "smaller_question":
            "{a}, {b} — यीमध्ये को होचो छ?",
    },


    "age": {

        "greater": [
            "{a} को उमेर {b} को उमेर भन्दा बढी छ।",
            "{b} सँग तुलना गर्दा {a} को उमेर बढी छ।",
        ],

        "reverse": [
            "{b} को उमेर {a} को उमेर भन्दा कम छ।",
            "{a} सँग तुलना गर्दा {b} को उमेर कम छ।",
        ],

        "greater_question":
            "{a}, {b} — यीमध्ये कसको उमेर बढी छ?",

        "smaller_question":
            "{a}, {b} — यीमध्ये कसको उमेर कम छ?",
    },


    "marks": {

        "greater": [
            "{a} ले {b} भन्दा बढी अंक ल्याए।",
            "{b} सँग तुलना गर्दा {a} ले बढी अंक ल्याए।",
        ],

        "reverse": [
            "{b} ले {a} भन्दा कम अंक ल्याए।",
            "{a} सँग तुलना गर्दा {b} ले कम अंक ल्याए।",
        ],

        "greater_question":
            "{a}, {b} — यीमध्ये कसले बढी अंक ल्याए?",

        "smaller_question":
            "{a}, {b} — यीमध्ये कसले कम अंक ल्याए?",
    },


    "speed": {

        "greater": [
            "{a} {b} भन्दा छिटो दौडे।",
            "{b} सँग तुलना गर्दा {a} छिटो दौडे।",
        ],

        "reverse": [
            "{b} {a} भन्दा ढिलो दौडे।",
            "{a} सँग तुलना गर्दा {b} ढिलो दौडे।",
        ],

        "greater_question":
            "{a}, {b} — यीमध्ये को छिटो दौडे?",

        "smaller_question":
            "{a}, {b} — यीमध्ये को ढिलो दौडे?",
    },
}


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def relation_sentence(
    attribute,
    bigger,
    smaller,
    reverse=False
):
    """
    Logical meaning is always:

        bigger > smaller

    reverse=True only changes the Telugu wording.
    """

    relation = RELATIONS[attribute]

    if reverse:
        template = random.choice(
            relation["reverse"]
        )
    else:
        template = random.choice(
            relation["greater"]
        )

    return template.format(
        a=bigger,
        b=smaller
    )


def make_question(
    attribute,
    a,
    b,
    ask_greater=True
):

    relation = RELATIONS[attribute]

    if ask_greater:

        template = (
            relation["greater_question"]
        )

    else:

        template = (
            relation["smaller_question"]
        )

    return template.format(
        a=a,
        b=b
    )


# ============================================================
# T1
# EASY — DIRECT NUMERICAL COMPARISON
# ============================================================

def generate_t1():

    a, b = random.sample(
        NAMES,
        2
    )

    topic = random.choice([
        "age",
        "marks",
        "books",
        "money",
    ])


    if topic == "age":

        value_a, value_b = random.sample(
            range(18, 70),
            2
        )

        fact1 = (
            f"{a} को उमेर "
            f"{value_a} वर्ष हो."
        )

        fact2 = (
            f"{b} को उमेर "
            f"{value_b} वर्ष हो."
        )


        ask_greater = random.choice([
            True,
            False
        ])


        if ask_greater:

            question = (
                f"{a}, {b} — "
                f"यीमध्ये कसको उमेर बढी छ?"
            )

            answer = (
                a
                if value_a > value_b
                else b
            )

            question_type = "greater"


        else:

            question = (
                f"{a}, {b} — "
                f"यीमध्ये कसको उमेर कम छ?"
            )

            answer = (
                a
                if value_a < value_b
                else b
            )

            question_type = "smaller"


    elif topic == "marks":

        value_a, value_b = random.sample(
            range(20, 101),
            2
        )

        fact1 = (
            f"{a} ले {value_a} "
            f"अंक ल्याए."
        )

        fact2 = (
            f"{b} ले {value_b} "
            f"अंक ल्याए."
        )


        ask_greater = random.choice([
            True,
            False
        ])


        if ask_greater:

            question = (
                f"{a}, {b} — "
                f"यीमध्ये कसले बढी "
                f"अंक ल्याए?"
            )

            answer = (
                a
                if value_a > value_b
                else b
            )

            question_type = "greater"


        else:

            question = (
                f"{a}, {b} — "
                f"यीमध्ये कसले कम "
                f"अंक ल्याए?"
            )

            answer = (
                a
                if value_a < value_b
                else b
            )

            question_type = "smaller"


    elif topic == "books":

        value_a, value_b = random.sample(
            range(1, 101),
            2
        )

        fact1 = (
            f"{a} सँग {value_a} "
            f"वटा किताब छन्."
        )

        fact2 = (
            f"{b} सँग {value_b} "
            f"वटा किताब छन्."
        )


        ask_greater = random.choice([
            True,
            False
        ])


        if ask_greater:

            question = (
                f"{a}, {b} — "
                f"यीमध्ये कोसँग बढी "
                f"किताब छन्?"
            )

            answer = (
                a
                if value_a > value_b
                else b
            )

            question_type = "greater"


        else:

            question = (
                f"{a}, {b} — "
                f"यीमध्ये कोसँग कम "
                f"किताब छन्?"
            )

            answer = (
                a
                if value_a < value_b
                else b
            )

            question_type = "smaller"


    else:

        value_a, value_b = random.sample(
            range(100, 5001),
            2
        )

        fact1 = (
            f"{a} सँग {value_a} "
            f"रुपैयाँ छन्."
        )

        fact2 = (
            f"{b} सँग {value_b} "
            f"रुपैयाँ छन्."
        )


        ask_greater = random.choice([
            True,
            False
        ])


        if ask_greater:

            question = (
                f"{a}, {b} — "
                f"यीमध्ये कोसँग बढी "
                f"पैसा छ?"
            )

            answer = (
                a
                if value_a > value_b
                else b
            )

            question_type = "greater"


        else:

            question = (
                f"{a}, {b} — "
                f"यीमध्ये कोसँग कम "
                f"पैसा छ?"
            )

            answer = (
                a
                if value_a < value_b
                else b
            )

            question_type = "smaller"


    prompt = "\n".join([
        fact1,
        fact2,
        question
    ])


    # Simple logical identifier.
    # No hashing.
    problem_key = (
        "T1",
        topic,
        a,
        value_a,
        b,
        value_b,
        question_type
    )


    return (
        prompt,
        answer,
        problem_key
    )


# ============================================================
# T2
# EASY — GREATER / SMALLER / EQUAL
# ============================================================

def generate_t2():

    a, b = random.sample(
        NAMES,
        2
    )


    relation = random.choice([
        "greater",
        "smaller",
        "equal"
    ])


    if relation == "equal":

        value_a = random.randint(
            20,
            100
        )

        value_b = value_a

        answer = "बराबर"


    elif relation == "greater":

        value_b = random.randint(
            20,
            80
        )

        value_a = random.randint(
            value_b + 1,
            100
        )

        answer = "बढी"


    else:

        value_a = random.randint(
            20,
            80
        )

        value_b = random.randint(
            value_a + 1,
            100
        )

        answer = "कम"


    prompt = (
        f"{a} ले {value_a} अंक ल्याए.\n"
        f"{b} ले {value_b} अंक ल्याए.\n"
        f"{a} ले ल्याएको अंक "
        f"{b} ले ल्याएको भन्दा "
        f"बढी हो, कम हो, कि बराबर हो?"
    )


    problem_key = (
        "T2",
        a,
        value_a,
        b,
        value_b,
        relation
    )


    return (
        prompt,
        answer,
        problem_key
    )


# ============================================================
# T3
# HARD — 3 ENTITY TRANSITIVE
#
# A > B > C
# ============================================================

def generate_t3():

    a, b, c = random.sample(
        NAMES,
        3
    )


    attribute = random.choice(
        list(RELATIONS.keys())
    )


    fact1 = relation_sentence(
        attribute,
        a,
        b
    )


    fact2 = relation_sentence(
        attribute,
        b,
        c
    )


    ask_greater = random.choice([
        True,
        False
    ])


    question = make_question(
        attribute,
        a,
        c,
        ask_greater
    )


    answer = (
        a
        if ask_greater
        else c
    )


    question_type = (
        "greater"
        if ask_greater
        else "smaller"
    )


    prompt = "\n".join([
        fact1,
        fact2,
        question
    ])


    problem_key = (
        "T3",
        attribute,
        a,
        b,
        c,
        question_type
    )


    return (
        prompt,
        answer,
        problem_key
    )


# ============================================================
# T4
# HARD — 4 ENTITY MULTI-HOP
#
# A > B > C > D
# ============================================================

def generate_t4():

    a, b, c, d = random.sample(
        NAMES,
        4
    )


    attribute = random.choice(
        list(RELATIONS.keys())
    )


    facts = [

        relation_sentence(
            attribute,
            a,
            b
        ),

        relation_sentence(
            attribute,
            b,
            c
        ),

        relation_sentence(
            attribute,
            c,
            d
        ),
    ]


    # Mix order
    random.shuffle(
        facts
    )


    ask_greater = random.choice([
        True,
        False
    ])


    question = make_question(
        attribute,
        a,
        d,
        ask_greater
    )


    answer = (
        a
        if ask_greater
        else d
    )


    question_type = (
        "greater"
        if ask_greater
        else "smaller"
    )


    prompt = "\n".join(
        facts + [question]
    )


    problem_key = (
        "T4",
        attribute,
        a,
        b,
        c,
        d,
        question_type
    )


    return (
        prompt,
        answer,
        problem_key
    )


# ============================================================
# T5
# HARD — MULTI-HOP + DISTRACTORS
# ============================================================

def generate_t5():

    a, b, c, d = random.sample(
        NAMES,
        4
    )


    attribute = random.choice(
        list(RELATIONS.keys())
    )


    facts = [

        relation_sentence(
            attribute,
            a,
            b
        ),

        relation_sentence(
            attribute,
            b,
            c
        ),

        relation_sentence(
            attribute,
            c,
            d
        ),
    ]


    distractor_people = random.sample(
        [a, b, c, d],
        2
    )


    distractor_bank = [

        lambda person:
            f"{person} लाई निलो रङ मन पर्छ.",

        lambda person:
            f"{person} हिजो बजार गए.",

        lambda person:
            f"{person} ले बिहान चिया खाए.",

        lambda person:
            f"{person} लाई आँप मन पर्छ.",

        lambda person:
            f"{person} सँग रातो झोला छ.",

        lambda person:
            f"{person} ले आइतबार चलचित्र हेरे.",
    ]


    chosen_distractors = random.sample(
        distractor_bank,
        2
    )


    distractors = [

        chosen_distractors[0](
            distractor_people[0]
        ),

        chosen_distractors[1](
            distractor_people[1]
        )
    ]


    all_facts = (
        facts
        + distractors
    )


    random.shuffle(
        all_facts
    )


    ask_greater = random.choice([
        True,
        False
    ])


    question = make_question(
        attribute,
        a,
        d,
        ask_greater
    )


    answer = (
        a
        if ask_greater
        else d
    )


    question_type = (
        "greater"
        if ask_greater
        else "smaller"
    )


    prompt = "\n".join(
        all_facts + [question]
    )


    problem_key = (
        "T5",
        attribute,
        a,
        b,
        c,
        d,
        question_type
    )


    return (
        prompt,
        answer,
        problem_key
    )


# ============================================================
# T6
# HARD — MIXED DIRECTION
#
# Logical relation:
#
# A > B > C > D
#
# But wording directions differ.
# ============================================================

def generate_t6():

    a, b, c, d = random.sample(
        NAMES,
        4
    )


    attribute = random.choice(
        list(RELATIONS.keys())
    )


    # Guarantees at least one normal
    # and one reverse relation
    patterns = [

        [False, True, False],

        [True, False, True],

        [False, False, True],

        [True, False, False],
    ]


    directions = random.choice(
        patterns
    )


    facts = [

        relation_sentence(
            attribute,
            a,
            b,
            reverse=directions[0]
        ),

        relation_sentence(
            attribute,
            b,
            c,
            reverse=directions[1]
        ),

        relation_sentence(
            attribute,
            c,
            d,
            reverse=directions[2]
        ),
    ]


    random.shuffle(
        facts
    )


    ask_greater = random.choice([
        True,
        False
    ])


    question = make_question(
        attribute,
        a,
        d,
        ask_greater
    )


    answer = (
        a
        if ask_greater
        else d
    )


    question_type = (
        "greater"
        if ask_greater
        else "smaller"
    )


    prompt = "\n".join(
        facts + [question]
    )


    problem_key = (
        "T6",
        attribute,
        a,
        b,
        c,
        d,
        tuple(directions),
        question_type
    )


    return (
        prompt,
        answer,
        problem_key
    )


# ============================================================
# GENERATOR MAPPING
# ============================================================

GENERATORS = {

    "T1": generate_t1,

    "T2": generate_t2,

    "T3": generate_t3,

    "T4": generate_t4,

    "T5": generate_t5,

    "T6": generate_t6,
}


# ============================================================
# GLOBAL DUPLICATE PROTECTION
#
# IMPORTANT:
# These sets are GLOBAL across all splits.
#
# Therefore:
# train, validation and test cannot contain
# the same prompt or same generated problem.
# ============================================================

seen_prompts = set()

seen_problem_keys = set()


# ============================================================
# GENERATE ONE SPLIT
# ============================================================

def generate_split(
    split_name
):

    examples = []


    for template_id in TEMPLATES:

        generator = (
            GENERATORS[
                template_id
            ]
        )


        required = (
            PER_TEMPLATE[
                split_name
            ]
        )


        count = 0


        while count < required:

            (
                prompt,
                answer,
                problem_key
            ) = generator()


            # ----------------------------------------
            # Prevent exact duplicate prompt
            # ----------------------------------------

            if prompt in seen_prompts:
                continue


            # ----------------------------------------
            # Prevent same logical instantiated problem
            # ----------------------------------------

            if problem_key in seen_problem_keys:
                continue


            # Accept example
            seen_prompts.add(
                prompt
            )

            seen_problem_keys.add(
                problem_key
            )


            example = {

                "id":
                    f"ne_{split_name}_"
                    f"{template_id}_"
                    f"{count:04d}",

                "language":
                    "Nepali",

                "split":
                    split_name,

                "template":
                    template_id,

                "prompt":
                    prompt,

                "answer":
                    answer,

                "text": (
                    "प्रश्न:\n"
                    + prompt
                    + "\nजवाफ:\n"
                    + answer
                )
            }


            examples.append(
                example
            )


            count += 1


    random.shuffle(
        examples
    )


    return examples


# ============================================================
# GENERATE DATASET
# ============================================================

def generate_dataset():

    # Order is deterministic
    # because random seed is fixed.

    train = generate_split(
        "train"
    )

    validation = generate_split(
        "validation"
    )

    test = generate_split(
        "test"
    )


    return {

        "train":
            train,

        "validation":
            validation,

        "test":
            test,
    }


# ============================================================
# VERIFY DATASET
# ============================================================

def verify_dataset(
    data
):

    print(
        "\nRunning dataset checks...\n"
    )


    # ========================================================
    # CHECK 1 — SPLIT SIZES
    # ========================================================

    assert (
        len(data["train"])
        == 9600
    )

    assert (
        len(data["validation"])
        == 1200
    )

    assert (
        len(data["test"])
        == 1200
    )


    print(
        "✓ Split sizes are correct."
    )


    # ========================================================
    # CHECK 2 — TOTAL SIZE
    # ========================================================

    total = sum(
        len(split)
        for split in data.values()
    )


    assert (
        total
        == TOTAL_EXAMPLES
    )


    print(
        f"✓ Total examples = {total}"
    )


    # ========================================================
    # CHECK 3 — CORRECT COUNT PER TEMPLATE
    # ========================================================

    for split_name in data:

        counts = Counter(

            example["template"]

            for example
            in data[split_name]
        )


        for template in TEMPLATES:

            expected = (
                PER_TEMPLATE[
                    split_name
                ]
            )


            assert (
                counts[template]
                == expected
            ), (
                f"{split_name} "
                f"{template}: "
                f"expected {expected}, "
                f"got {counts[template]}"
            )


    print(
        "✓ Every template has the "
        "correct number of examples."
    )


    # ========================================================
    # CHECK 4 — NO DUPLICATE PROMPTS INSIDE OR ACROSS SPLITS
    # ========================================================

    train_prompts = {

        x["prompt"]

        for x in data["train"]
    }


    val_prompts = {

        x["prompt"]

        for x in data["validation"]
    }


    test_prompts = {

        x["prompt"]

        for x in data["test"]
    }


    # Inside each split
    assert (
        len(train_prompts)
        == len(data["train"])
    )

    assert (
        len(val_prompts)
        == len(data["validation"])
    )

    assert (
        len(test_prompts)
        == len(data["test"])
    )


    # Between splits
    assert train_prompts.isdisjoint(
        val_prompts
    )

    assert train_prompts.isdisjoint(
        test_prompts
    )

    assert val_prompts.isdisjoint(
        test_prompts
    )


    print(
        "✓ No prompt is duplicated "
        "inside or across splits."
    )


    # ========================================================
    # CHECK 5 — UNIQUE IDS
    # ========================================================

    all_ids = [

        example["id"]

        for split in data.values()

        for example in split
    ]


    assert (
        len(all_ids)
        == len(set(all_ids))
    )


    print(
        "✓ Every example ID is unique."
    )


    # ========================================================
    # CHECK 6 — NAMES ARE INTENTIONALLY SHARED
    #
    # This is NOT leakage.
    # We explicitly confirm the design.
    # ========================================================

    print(
        "✓ Shared entity-name pool is intentional."
    )


    print(
        "\n======================================"
    )

    print(
        "ALL DATASET CHECKS PASSED"
    )

    print(
        "======================================\n"
    )


# ============================================================
# SAVE JSONL
# ============================================================

def save_jsonl(
    filename,
    examples
):

    with open(
        filename,
        "w",
        encoding="utf-8"
    ) as f:

        for example in examples:

            f.write(

                json.dumps(
                    example,
                    ensure_ascii=False
                )

                + "\n"
            )


# ============================================================
# SAVE DATASET
# ============================================================

def save_dataset(
    data,
    output_dir
):

    output_dir = Path(
        output_dir
    )


    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    save_jsonl(
        output_dir
        / "train.jsonl",

        data["train"]
    )


    save_jsonl(
        output_dir
        / "validation.jsonl",

        data["validation"]
    )


    save_jsonl(
        output_dir
        / "test.jsonl",

        data["test"]
    )


    # Save simple dataset statistics
    stats = {

        "total":
            12000,

        "train":
            9600,

        "validation":
            1200,

        "test":
            1200,

        "templates":
            TEMPLATES,

        "shared_name_pool":
            True,

        "exact_prompt_overlap":
            False,

        "seed":
            SEED
    }


    with open(
        output_dir
        / "dataset_stats.json",

        "w",

        encoding="utf-8"
    ) as f:

        json.dump(
            stats,
            f,
            ensure_ascii=False,
            indent=2
        )


# ============================================================
# PRINT SAMPLE FROM EACH TEMPLATE
# ============================================================

def print_samples(
    data
):

    for template in TEMPLATES:

        example = next(

            x
            for x in data["train"]

            if x["template"]
            == template
        )


        print(
            "\n" + "=" * 70
        )

        print(
            template
        )

        print(
            "=" * 70
        )

        print(
            example["prompt"]
        )

        print(
            "\nजवाफ:",
            example["answer"]
        )


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":

    random.seed(
        SEED
    )


    print(
        "Generating revised Nepali "
        "reasoning dataset..."
    )


    data = (
        generate_dataset()
    )


    verify_dataset(
        data
    )


    OUTPUT_DIR = (
        "/kaggle/working/"
        "nepali_reasoning_dataset_shared_names"
    )


    save_dataset(
        data,
        OUTPUT_DIR
    )


    print_samples(
        data
    )


    print(
        "\nDataset saved to:"
    )

    print(
        OUTPUT_DIR
    )

Generating revised Nepali reasoning dataset...

Running dataset checks...

✓ Split sizes are correct.
✓ Total examples = 12000
✓ Every template has the correct number of examples.
✓ No prompt is duplicated inside or across splits.
✓ Every example ID is unique.
✓ Shared entity-name pool is intentional.

ALL DATASET CHECKS PASSED


T1
सुरेश सँग 5 वटा किताब छन्.
पवन सँग 12 वटा किताब छन्.
सुरेश, पवन — यीमध्ये कोसँग बढी किताब छन्?

जवाफ: पवन

T2
तरुण ले 97 अंक ल्याए.
संदीप ले 59 अंक ल्याए.
तरुण ले ल्याएको अंक संदीप ले ल्याएको भन्दा बढी हो, कम हो, कि बराबर हो?

जवाफ: बढी

T3
वर्षा सँग तुलना गर्दा सुनिल अग्लो छन्।
वर्षा दीपक भन्दा अग्लो छन्।
सुनिल, दीपक — यीमध्ये को होचो छ?

जवाफ: दीपक

T4
तरुण को उमेर कार्तिक को उमेर भन्दा बढी छ।
आदित्य को उमेर वर्षा को उमेर भन्दा बढी छ।
आदित्य सँग तुलना गर्दा कार्तिक को उमेर बढी छ।
तरुण, वर्षा — यीमध्ये कसको उमेर कम छ?

जवाफ: वर्षा

T5
सुनिल को उमेर लता को उमेर भन्दा बढी छ।
सुनिल सँग तुलना गर्दा आदित्य को उमेर बढी छ।
संदीप हिजो बजार गए.
आदित्य सँग तुलना गर्